# 🔍 Deblur/Unblur Image - Training on Google Colab
**Đồ án TGMT - 3 cấp độ: Face, Scene, ID Card**

Notebook này sẽ:
1. Clone mã nguồn dự án từ GitHub của bạn
2. Mount Google Drive để lưu model an toàn
3. Tải dataset (ví dụ: GoPro cho Scene)
4. Train model SwinIR
5. Lưu kết quả về Google Drive

## 1️⃣ Kiểm tra GPU & Mount Google Drive
Việc Mount Drive là BẮT BUỘC để lưu trữ model checkpoint. Colab sẽ tự xóa dữ liệu khi bạn tắt tab.

In [ ]:
!nvidia-smi
print('---')
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2️⃣ Tải Mã Nguồn Từ GitHub
Chúng ta sẽ kéo (clone) code trực tiếp từ kho GitHub của bạn. Bất cứ khi nào bạn sửa code trên máy tính và push lên Github, bạn chỉ cần chạy lại cell này để lấy code mới nhất.

In [ ]:
import os

# Luôn quay về thư mục gốc của Colab trước khi xử lý
%cd /content

# Xóa thư mục cũ (nếu bạn chạy lại cell này để cập nhật code)
!rm -rf ThiGiacMayTinh

# Clone code từ GitHub của bạn
!git clone https://github.com/hnihTyoB/ThiGiacMayTinh.git

# Di chuyển vào thư mục dự án
%cd ThiGiacMayTinh

# Cài đặt các thư viện cần thiết
!pip install -q basicsr lmdb pyyaml tb-nightly tqdm gdown

print('\n[✓] Dự án đã sẵn sàng!')

## 3️⃣ Tải Dataset
### Scene: GoPro Large (~5.6GB)

In [ ]:
# Chạy script tải và chuẩn bị GoPro dataset tự động
!python scripts/data_preparation/download_deblur_datasets.py --scene

### Face & ID Card (Tuỳ chọn)
Nếu bạn train Face hoặc ID Card, hãy upload ảnh gốc (sharp) lên thư mục Drive của bạn, sau đó copy vào môi trường Colab và chạy script tạo blur.

In [ ]:
import os

# Hướng dẫn tải Face/ID Card data:
# 1. Upload ảnh sharp lên: /content/drive/MyDrive/face_sharp/
# 2. Copy vào Colab:
# !mkdir -p datasets/face/train/sharp/
# !cp -r '/content/drive/MyDrive/face_sharp/'* datasets/face/train/sharp/
# 3. Chạy lệnh tạo blur:
# !python scripts/data_preparation/create_blur_dataset.py --input datasets/face/train/sharp --output datasets/face/train/blur --task face

## 4️⃣ Training
Bỏ comment (`#`) ở lệnh bạn muốn chạy. Mặc định đang train Scene.

In [ ]:
# ========== TRAIN SCENE DEBLUR ==========
!python basicsr/train.py -opt options/train/train_deblur_scene.yml

# ========== TRAIN FACE DEBLUR ==========
# !python basicsr/train.py -opt options/train/train_deblur_face.yml

# ========== TRAIN ID CARD DEBLUR ==========
# !python basicsr/train.py -opt options/train/train_deblur_idcard.yml

## 5️⃣ Lưu Model Về Google Drive
**RẤT QUAN TRỌNG:** Chạy cell này để copy các checkpoint `.pth` đã train từ Colab sang tài khoản Google Drive của bạn (để tải về máy local dùng sau này).

In [ ]:
import shutil, os

# Thư mục trên Drive để lưu model
save_dir = '/content/drive/MyDrive/DeblurModels'
os.makedirs(save_dir, exist_ok=True)

if os.path.exists('experiments'):
    for exp in os.listdir('experiments'):
        src = f'experiments/{exp}/models'
        if os.path.exists(src):
            dst = f'{save_dir}/{exp}'
            os.makedirs(dst, exist_ok=True)
            for f in os.listdir(src):
                shutil.copy2(f'{src}/{f}', f'{dst}/{f}')
                print(f'  Saved: {dst}/{f}')

print(f'\n[✓] Tất cả model đã được sao lưu an toàn tại thư mục DeblurModels trên Google Drive!')

## 6️⃣ Test & Inference (Tùy chọn)
Kiểm tra thử độ hiệu quả trực tiếp trên Colab.

In [ ]:
import os
os.makedirs('test_images', exist_ok=True)
os.makedirs('results', exist_ok=True)

TASK = 'scene'
MODEL_PATH = 'experiments/DeblurScene_SwinIR/models/net_g_latest.pth'

# Upload 1-2 ảnh mờ vào thư mục test_images/ bên thanh bên trái trước khi chạy lệnh này
if os.path.exists(MODEL_PATH):
    !python inference/inference_deblur.py \
        --input test_images \
        --output results \
        --model_path {MODEL_PATH} \
        --task {TASK}
else:
    print(f'⚠️ Chưa có model tại: {MODEL_PATH}. Hãy train trước!')